In [3]:
using Random

In [ ]:

function holdOut(N::Int, P::Real)
    @assert 0 <= P <= 1 "P debe estar entre 0 y 1"
    indices = randperm(N)  # Generate a random permutation of indexes
    split_point = floor(Int, N * (1 - P))  # Determines cut-off point
    return (indices[1:split_point], indices[split_point+1:end])  # Returns the tuple with the subdatasets
end

holdOut (generic function with 1 method)

In [6]:
sets = holdOut(150, 0.3)
typeof(sets)


Tuple{Vector{Int64}, Vector{Int64}}

In [8]:

function holdOut(N::Int, Pval::Real, Ptest::Real)
    @assert 0 <= Pval + Ptest <= 1 "La suma de Pval y Ptest debe estar entre 0 y 1"
    (train_test, test) = holdOut(N, Ptest)  # Separates the test dataset
    Pval_adjusted = Pval / (1 - Ptest)  # Adjusts Pval for the remaining datasets 
    train, val = holdOut(length(train_test), Pval_adjusted)  # Separates validate dataset from training dataset
    return (train_test[train], train_test[val], test)  # Returns the three datasets
end

holdOut (generic function with 2 methods)

In [12]:
(a, b, c) = holdOut(150, 0.15, 0.15)
x = length(a) + length(b) + length(c)
typeof((a, b, c))


Tuple{Vector{Int64}, Vector{Int64}, Vector{Int64}}

In [ ]:

function trainClassANN(topology:: AbstractArray{<: Int, 1}, dataset:: Tuple{AbstractArray{<: Real, 2}, AbstractArray{Bool, 2}};
    transferFunctions::AbstractArray{<:Function,1} = fill(σ, length(topology)), maxEpochs:: Int = 1000, minLoss:: Real = 0.0, learningRate:: Real = 0.01)

   inputs, targets = dataset
   inputs = Float32.(inputs')  # Convertir a Float32 y trasponer
   targets = (targets')   # Trasponer
   
   numInputs = size(inputs, 1)
   numOutputs = size(targets, 1)
   
   ann = buildClassANN(numInputs, topology, numOutputs; transferFunctions) # build ANN
   loss(model, x,y) = (size(y,1) == 1) ? Losses.binarycrossentropy(model(x),y) : Losses.crossentropy(model(x),y) # loss function
   opt_state = Flux.setup(ADAM(learningRate), ann) # optimizer

   losses = Float32[] # losses array
   push!(losses, loss(ann, inputs, targets)) # append iteration 0 loss 
   
   for epoch in 1:maxEpochs
       
       Flux.train!(loss, ann, [(inputs, targets)], opt_state)
       current_loss = loss(ann, inputs, targets)
       push!(losses, current_loss)
       
       if current_loss ≤ minLoss
           break
       end
   end
   
   return ann, losses
end

In [ ]:
function trainClassANN(topology:: AbstractArray{<: Int, 1}, (inputs, targets):: Tuple{AbstractArray{<: Real, 2}, AbstractArray{Bool, 1}};
    transferFunctions::AbstractArray{<:Function,1} = fill(σ, length(topology)), maxEpochs:: Int = 1000, minLoss:: Real = 0.0, learningRate:: Real = 0.01)

   reshape!(targets, :, 1) 
   return trainClassANN(topology, (inputs, targets); transferFunctions=transferFunctions, maxEpochs=maxEpochs, minLoss=minLoss, learningRate=learningRate)
   
end

In [ ]:
function trainClassANN(topology::AbstractArray{<:Int,1},
    trainingDataset::  Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}};
    validationDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}=(Array{eltype(trainingDataset[1]),2}(undef,0,size(trainingDataset[1],2)), falses(0,size(trainingDataset[2],2))),
    testDataset::      Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}=(Array{eltype(trainingDataset[1]),2}(undef,0,size(trainingDataset[1],2)), falses(0,size(trainingDataset[2],2))),
    transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)),
    maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01, maxEpochsVal::Int=20)
    
    trainingInputs = Float32.(trainingDataset[1]')
    validationInputs = Float32.(validationDataset[1]')
    testInputs = Float32.(testDataset[1]')

    trainingTargets = trainingDataset[2]'
    validationTargets = validationDataset[2]'
    testTargets = testDataset[2]'
    
    numTrainingInputs = size(trainingInputs, 1)
    numTrainingOutputs = size(trainingTargets, 1)
    
    ann = buildClassANN(numTrainingInputs, topology, numTrainingOutputs; transferFunctions) # build ANN
    loss(model, x,y) = (size(y,1) == 1) ? Losses.binarycrossentropy(model(x),y) : Losses.crossentropy(model(x),y) # loss function
    opt_state = Flux.setup(ADAM(learningRate), ann) # optimizer

    trainingLosses = Float32[] # losses array
    validationLosses = Float32[]
    testLosses = Float32[]

    push!(trainingLosses, loss(ann, trainingInputs, trainingTargets)) # append iteration 0 loss 
    push!(validationLosses, loss(ann, validationInputs, validationTargets))
    push!(testLosses, loss(ann, testInputs, testTargets))

    bestValidationLoss = Inf
    bestAnn = deepcopy(ann)
    epochsWithoutImprovement = 0

    for epoch in 1:maxEpochs
        # ENTRENAMIENTO
        Flux.train!(loss, ann, [(trainingInputs, trainingTargets)], opt_state)
        currentTrainingLoss = loss(ann, trainingInputs, trainingTargets)
        push!(trainingLosses, currentTrainingLoss)
        
        if currentTrainingLoss <= minLoss
            break
        end
    
        # VALIDACIÓN (si hay conjunto de validación)
        if !isempty(validationInputs)
            currentValidationLoss = loss(ann, validationInputs, validationTargets)
            push!(validationLosses, currentValidationLoss)
            
            # PARADA TEMPRANA: Si el loss en validación mejora, actualizar mejor modelo
            if currentValidationLoss < bestValidationLoss
                bestValidationLoss = currentValidationLoss
                bestAnn = deepcopy(ann)
                epochsWithoutImprovement = 0  # Reiniciar el contador de epochs sin mejora
            else
                epochsWithoutImprovement += 1
            end
        end
    
        # PRUEBA (si hay conjunto de test)
        if !isempty(testInputs)
            currentTestLoss = loss(ann, testInputs, testTargets)
            push!(testLosses, currentTestLoss)
        end
    
        # CRITERIO DE PARADA TEMPRANA
        if epochsWithoutImprovement >= maxEpochsVal
            println("Parada temprana en epoch $epoch")
            break
        end
    end
    if !isempty(validationInputs)
        return bestAnn, trainingLosses, validationLosses, testLosses
    else
        return ann, trainingLosses, validationLosses, testLosses
    end
end;

In [ ]:
function trainClassANN(topology::AbstractArray{<:Int,1},
    trainingDataset::  Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}};
    validationDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}=(Array{eltype(trainingDataset[1]),2}(undef,0,size(trainingDataset[1],2)), falses(0)),
    testDataset::      Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}}=(Array{eltype(trainingDataset[1]),2}(undef,0,size(trainingDataset[1],2)), falses(0)),
    transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)),
    maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01, maxEpochsVal::Int=20)

    newTrainingDataset = (trainingDataset[1], reshape(trainingDataset[2], :, 1))
    newValidationDataset = (validationDataset[1], reshape(validationDataset[2], :, 1))
    newTestDataset = (testDataset[1], reshape(testDataset[2], :, 1))

    return trainClassANN(topology, newTrainingDataset; validationDataset=newValidationDataset, testDataset=newTestDataset, 
                         transferFunctions=transferFunctions, maxEpochs=maxEpochs, minLoss=minLoss, 
                         learningRate=learningRate, maxEpochsVal=maxEpochsVal)
end;